In [52]:
!pip install pytorch-crf -q

In [53]:
from torchcrf import CRF

In [54]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
import random
import numpy as np

In [55]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [56]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [57]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [58]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [59]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

print("Sentence data created.")

for dialect in sentence_data:
    print(dialect, len(sentence_data[dialect]))

Sentence data created.
Barishal 3481
Chittagong 3481
Mymensingh 3481
Noakhali 3481
Sylhet 3481


In [60]:
import json

with open(
    "/kaggle/input/datasets/arnav1605/ancholink-ner/within_dialect_splits.json",
    "r"
) as f:
    splits = json.load(f)

print("Splits loaded.")

Splits loaded.


In [61]:
tags = set()

for sentences in sentence_data.values():
    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag for tag, i in tag_to_id.items()
}

print("Number of tags:", len(tags))
print(tag_to_id)

Number of tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [62]:
def build_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for token in sentence["tokens"]:

            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab

In [63]:
class CharNERDataset(Dataset):

    def __init__(self, sentences, word_vocab, char_vocab):

        self.data = []

        for sentence in sentences:

            word_ids, tag_ids = encode_sentence(
                sentence,
                word_vocab
            )

            char_ids = encode_characters(
                sentence,
                char_vocab
            )

            self.data.append(
                (word_ids, char_ids, tag_ids)
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        words, chars, tags = self.data[index]

        return words, chars, tags

In [64]:
class NERDataset(Dataset):

    def __init__(self, sentences, vocab):

        self.data = [
            encode_sentence(sentence, vocab)
            for sentence in sentences
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        tokens, tags = self.data[index]

        return (
            torch.tensor(tokens, dtype=torch.long),
            torch.tensor(tags, dtype=torch.long)
        )

In [65]:
def char_collate_fn(batch):

    words, chars, tags = zip(*batch)

    max_sentence_length = max(
        len(x) for x in words
    )

    max_word_length = max(
        len(char)
        for sentence in chars
        for char in sentence
    )

    padded_words = []
    padded_tags = []
    padded_chars = []
    masks = []

    for word_ids, char_ids, tag_ids in zip(
        words, chars, tags
    ):

        sentence_length = len(word_ids)

        # Word padding
        word_padding = (
            max_sentence_length - sentence_length
        )

        padded_words.append(
            word_ids +
            [0] * word_padding
        )

        # Tag padding
        padded_tags.append(
            tag_ids +
            [0] * word_padding
        )

        # Sentence mask
        masks.append(
            [1] * sentence_length +
            [0] * word_padding
        )

        # Character padding
        sentence_chars = []

        for word in char_ids:

            char_padding = (
                max_word_length - len(word)
            )

            sentence_chars.append(
                word +
                [0] * char_padding
            )

        # Add completely padded words
        for _ in range(word_padding):

            sentence_chars.append(
                [0] * max_word_length
            )

        padded_chars.append(sentence_chars)

    return (
        torch.tensor(padded_words, dtype=torch.long),
        torch.tensor(padded_chars, dtype=torch.long),
        torch.tensor(padded_tags, dtype=torch.long),
        torch.tensor(masks, dtype=torch.bool)
    )

In [66]:
def collate_fn(batch):

    tokens, tags = zip(*batch)

    max_length = max(len(x) for x in tokens)

    padded_tokens = []
    padded_tags = []
    masks = []

    for x, y in zip(tokens, tags):

        length = len(x)
        padding = max_length - length

        padded_tokens.append(
            torch.cat([
                x,
                torch.zeros(padding, dtype=torch.long)
            ])
        )

        padded_tags.append(
            torch.cat([
                y,
                torch.zeros(padding, dtype=torch.long)
            ])
        )

        masks.append(
            torch.tensor(
                [1] * length + [0] * padding,
                dtype=torch.bool
            )
        )

    return (
        torch.stack(padded_tokens),
        torch.stack(padded_tags),
        torch.stack(masks)
    )

In [67]:
vocabularies = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    vocabularies[dialect] = build_vocab(train_sentences)

    print(
        dialect,
        "| Vocabulary size:", len(vocabularies[dialect])
    )

Barishal | Vocabulary size: 3371
Chittagong | Vocabulary size: 4317
Mymensingh | Vocabulary size: 3467
Noakhali | Vocabulary size: 4053
Sylhet | Vocabulary size: 3845


In [68]:
char_vocabularies = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    char_vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in train_sentences:

        for word in sentence["tokens"]:

            for char in word:

                if char not in char_vocab:
                    char_vocab[char] = len(char_vocab)

    char_vocabularies[dialect] = char_vocab

    print(
        dialect,
        "| Characters:",
        len(char_vocab)
    )

Barishal | Characters: 69
Chittagong | Characters: 76
Mymensingh | Characters: 69
Noakhali | Characters: 69
Sylhet | Characters: 77


In [69]:
def encode_characters(sentence, char_vocab):

    word_chars = []

    for word in sentence["tokens"]:

        chars = [
            char_vocab.get(char, char_vocab["<UNK>"])
            for char in word
        ]

        word_chars.append(chars)

    return word_chars

In [70]:
def encode_sentence(sentence, vocab):

    token_ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in sentence["tokens"]
    ]

    tag_ids = [
        tag_to_id[tag]
        for tag in sentence["tags"]
    ]

    return token_ids, tag_ids

In [71]:
class CharCNNBiLSTMCRF(nn.Module):

    def __init__(
        self,
        word_vocab_size,
        char_vocab_size,
        tag_size,
        word_embedding_dim=100,
        char_embedding_dim=30,
        char_channels=50,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            word_vocab_size,
            word_embedding_dim,
            padding_idx=0
        )

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            char_embedding_dim,
            padding_idx=0
        )

        self.char_cnn = nn.Conv1d(
            in_channels=char_embedding_dim,
            out_channels=char_channels,
            kernel_size=3,
            padding=1
        )

        self.lstm = nn.LSTM(
            input_size=word_embedding_dim + char_channels,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(self, words, chars, tags=None, mask=None):

        # Word representation
        word_emb = self.word_embedding(words)

        # Character representation
        char_emb = self.char_embedding(chars)

        batch_size = chars.size(0)
        sentence_length = chars.size(1)
        word_length = chars.size(2)

        # Combine batch and sentence dimensions
        char_emb = char_emb.view(
            batch_size * sentence_length,
            word_length,
            -1
        )

        # Conv1D expects:
        # batch × channels × sequence
        char_emb = char_emb.transpose(1, 2)

        char_features = self.char_cnn(char_emb)

        # Max pooling over characters
        char_features = torch.max(
            char_features,
            dim=2
        ).values

        # Restore sentence structure
        char_features = char_features.view(
            batch_size,
            sentence_length,
            -1
        )

        # Combine word + character representations
        x = torch.cat(
            [word_emb, char_features],
            dim=2
        )

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            loss = -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

            return loss

        predictions = self.crf.decode(
            emissions,
            mask=mask
        )

        return predictions

In [72]:
class BiGRUCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size,
        embedding_dim=100,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(self, tokens, tags=None, mask=None):

        x = self.embedding(tokens)

        x, _ = self.gru(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            loss = -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

            return loss

        predictions = self.crf.decode(
            emissions,
            mask=mask
        )

        return predictions

In [73]:
char_train_loaders = {}
char_test_loaders = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])
    test_ids = set(splits[dialect]["test"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    test_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in test_ids
    ]

    train_dataset = CharNERDataset(
        train_sentences,
        vocabularies[dialect],
        char_vocabularies[dialect]
    )

    test_dataset = CharNERDataset(
        test_sentences,
        vocabularies[dialect],
        char_vocabularies[dialect]
    )

    char_train_loaders[dialect] = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=char_collate_fn
    )

    char_test_loaders[dialect] = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=char_collate_fn
    )

    print(
        dialect,
        "| Train:", len(train_dataset),
        "| Test:", len(test_dataset)
    )

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [74]:
def train_char_model(model, train_loader, epochs=20, lr=1e-3):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for words, chars, tags, mask in train_loader:

            words = words.to(device)
            chars = chars.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                words,
                chars,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {average_loss:.4f}"
        )

In [75]:
def get_entities(tags):

    entities = []
    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):

        if tag.startswith("B-"):

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):

            if start is None:
                start = i
                entity_type = tag[2:]

        else:

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = None
            entity_type = None

    return set(entities)

In [76]:
def evaluate_char_model(model, test_loader):

    model.eval()

    total_correct = 0
    total_predicted = 0
    total_actual = 0

    with torch.no_grad():

        for words, chars, tags, mask in test_loader:

            words = words.to(device)
            chars = chars.to(device)
            mask = mask.to(device)

            predictions = model(
                words,
                chars,
                mask=mask
            )

            for i, pred in enumerate(predictions):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(true_tags)
                pred_entities = get_entities(pred_tags)

                total_correct += len(
                    true_entities & pred_entities
                )

                total_actual += len(true_entities)
                total_predicted += len(pred_entities)

    precision = (
        total_correct / total_predicted
        if total_predicted > 0 else 0
    )

    recall = (
        total_correct / total_actual
        if total_actual > 0 else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    return precision, recall, f1

In [77]:
char_models = {}
char_results = {}

for dialect in sentence_data:

    print("\n" + "=" * 50)
    print("Training:", dialect)
    print("=" * 50)

    model = CharCNNBiLSTMCRF(
        word_vocab_size=len(vocabularies[dialect]),
        char_vocab_size=len(char_vocabularies[dialect]),
        tag_size=len(tag_to_id)
    ).to(device)

    train_char_model(
        model,
        char_train_loaders[dialect],
        epochs=20,
        lr=1e-3
    )

    precision, recall, f1 = evaluate_char_model(
        model,
        char_test_loaders[dialect]
    )

    char_models[dialect] = model

    char_results[dialect] = {
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

    print(
        f"\n{dialect} Results:"
        f"\nPrecision: {precision:.4f}"
        f"\nRecall:    {recall:.4f}"
        f"\nF1:        {f1:.4f}"
    )


Training: Barishal
Epoch 1/20 - Loss: 4.5250
Epoch 2/20 - Loss: 2.5552
Epoch 3/20 - Loss: 1.8254
Epoch 4/20 - Loss: 1.3408
Epoch 5/20 - Loss: 1.0220
Epoch 6/20 - Loss: 0.7663
Epoch 7/20 - Loss: 0.5800
Epoch 8/20 - Loss: 0.4164
Epoch 9/20 - Loss: 0.3014
Epoch 10/20 - Loss: 0.2201
Epoch 11/20 - Loss: 0.1491
Epoch 12/20 - Loss: 0.1125
Epoch 13/20 - Loss: 0.0851
Epoch 14/20 - Loss: 0.0611
Epoch 15/20 - Loss: 0.0470
Epoch 16/20 - Loss: 0.0383
Epoch 17/20 - Loss: 0.0342
Epoch 18/20 - Loss: 0.0349
Epoch 19/20 - Loss: 0.0270
Epoch 20/20 - Loss: 0.0235

Barishal Results:
Precision: 0.8381
Recall:    0.6633
F1:        0.7405

Training: Chittagong
Epoch 1/20 - Loss: 4.8202
Epoch 2/20 - Loss: 2.8457
Epoch 3/20 - Loss: 2.1647
Epoch 4/20 - Loss: 1.6827
Epoch 5/20 - Loss: 1.3473
Epoch 6/20 - Loss: 1.0562
Epoch 7/20 - Loss: 0.8211
Epoch 8/20 - Loss: 0.6170
Epoch 9/20 - Loss: 0.4593
Epoch 10/20 - Loss: 0.3309
Epoch 11/20 - Loss: 0.2316
Epoch 12/20 - Loss: 0.1695
Epoch 13/20 - Loss: 0.1255
Epoch 14/20 

In [78]:
char_results_table = pd.DataFrame(
    char_results
).T

char_results_table

,Precision,Recall,F1
Barishal,0.838095,0.663317,0.740533
Chittagong,0.796610,0.602564,0.686131
Mymensingh,0.863636,0.673418,0.756757
Noakhali,0.774648,0.556962,0.648012
Sylhet,0.806962,0.639098,0.713287
